# Graph of thoughts

While Tree of Thoughts enables exploration of multiple reasoning paths through a hierarchical structure, many complex problems don't follow a purely tree-like pattern. Real-world reasoning often involves circular dependencies, multiple paths leading to the same conclusion, combining insights from different branches, or revisiting earlier thoughts with new information. These scenarios require a more flexible structure than a tree can provide.

Graph of Thoughts (GoT) extends the tree-based approach by modeling reasoning as an arbitrary graph where thoughts are nodes and relationships between thoughts are edges. This allows for much richer reasoning patterns: thoughts can reference multiple predecessors, independent reasoning branches can merge, insights can flow backwards to refine earlier thoughts, and sub-problems can be solved in parallel then combined. The graph structure naturally captures the non-linear, interconnected nature of human problem-solving.

In this notebook, we will implement a Graph of Thoughts system that enables flexible, non-linear reasoning. We will build a framework where thoughts can have multiple dependencies, results can be aggregated from parallel paths, and the reasoning structure adapts to the problem's inherent complexity rather than forcing it into a tree hierarchy. This approach is particularly valuable for problems requiring synthesis of multiple perspectives, modular decomposition with shared sub-problems or iterative refinement based on feedback loops.

In [1]:
import os
import json
from typing import List, Tuple, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model

In [2]:
# Initialize the language model — used for both generating and evaluating thoughts
llm = ChatOpenAI(
    model="gpt-4o-mini-2024-07-18",
    api_key=os.getenv("OPENAI_API_KEY", "").strip(),
    temperature=0.7  # Moderate temperature: enough diversity for brainstorming, enough consistency for evaluation
)

The same model and temperature used for Tree of Thoughts work equally well here. Generation benefits from some diversity (`0.7` helps); aggregation and evaluation both tolerate it. Using a single shared instance for all three roles keeps the code simple.

## Define the ThoughtNode
The only structural difference between a tree node and a graph node is that a tree node has exactly one parent while a graph node can have several. In this implementation `parents` is a list of `ThoughtNode` objects - empty for the root, one element for nodes produced by `generate` or `refine`, and multiple elements for nodes produced by `aggregate`. The constructor registers the new node as a child of every parent, so both directions of each edge are tracked without any extra bookkeeping.

Depth is defined as one more than the maximum depth among all parents. For single-parent nodes this is identical to the tree definition. For aggregate nodes - whose parents may come from different depths, since independent branches may have expanded by different amounts - taking the maximum ensures that depth always reflects the longest chain of reasoning feeding into this node.

Each node also records an `operation` string: `"root"`, `"generate"`, `"aggregate"`, or `"refine"`. This field is used in log output and the graph visualization to show not just *what* was thought but *how* that thought came to exist.

In [3]:
class ThoughtNode:
    """A single reasoning step in the graph. Can have multiple parents, unlike a tree node."""

    _counter = 0  # class-level counter; incremented each time a node is created

    @classmethod
    def reset_counter(cls):
        """Reset IDs to 0 before starting a new run so Node #1 is always the root."""
        cls._counter = 0

    def __init__(
        self,
        thought: str,
        parents: Optional[List['ThoughtNode']] = None,
        score: float = 0.0,
        operation: str = "generate"   # "root" | "generate" | "aggregate" | "refine"
    ):
        ThoughtNode._counter += 1
        self.node_id = ThoughtNode._counter       # unique sequential ID for log tracing
        self.thought = thought                     # text content of this reasoning step
        self.parents: List['ThoughtNode'] = parents or []   # empty for root; many for aggregate
        self.children: List['ThoughtNode'] = []
        self.score = score                         # LLM-assigned evaluation score 0–10
        self.operation = operation                 # which GoT operation produced this node

        # Depth: root is 0; one more than the deepest parent.
        # max() is correct for aggregate nodes whose parents may sit at different depths.
        self.depth = 0 if not self.parents else max(p.depth for p in self.parents) + 1

        # Register this node as a child of every parent — keeps both edge directions in sync
        for p in self.parents:
            p.children.append(self)

    def get_path(self) -> List[str]:
        """
        Reconstruct a root-to-node chain by following the first parent at each level.
        For aggregate nodes this traces back through the first branch — a useful approximation.
        """
        path = []
        current = self
        while current is not None:
            path.append(current.thought)
            # Follow first parent upward; aggregate nodes have many but we pick one chain
            current = current.parents[0] if current.parents else None
        return list(reversed(path))  # collected leaf-to-root, so reverse

**`ThoughtNode`** has two differences from its Tree of Thoughts counterpart.

- `parents` is a list, not a single reference. The constructor's `for p in self.parents: p.children.append(self)` loop handles any number of parents uniformly - one for `generate` and `refine` nodes, many for `aggregate` nodes.
- `depth` uses `max(p.depth for p in self.parents) + 1`. For aggregate nodes this selects the deepest incoming branch, so depth always measures the longest predecessor chain.

`get_path()` follows the *first* parent at each level. For aggregate nodes this traces back through the first branch merged - a reasonable approximation for display, since the full multi-branch ancestry is too wide to render as a single chain.

## The three GoT operations: generate, aggregate, refine
Tree of Thoughts had one way to extend the graph: take a node and ask the model for several candidate next steps. Graph of Thoughts keeps that operation and adds two more.
- **Generate** works exactly as in Tree of Thoughts: given a parent node, produce N candidate next thoughts. In GoT this is used to build the independent parallel branches - each branch is a chain of single-parent generate steps.
- **Aggregate** is the operation that makes GoT structurally different from a tree. It takes a list of nodes - typically the leaf nodes of independent parallel branches - and asks the model to synthesise them into one unified thought. The result becomes a new node whose `parents` list contains *all* the input nodes, creating a convergent multi-parent edge that is impossible in a tree.
- **Refine** takes a single node and produces one improved version of it. It creates a single-parent edge just like `generate`, but instead of exploring a new direction it tightens the existing thought - making it more specific, filling gaps, correcting weaknesses. Used in a short loop after aggregation, it incrementally sharpens the combined result.

We implement the operations in the order they appear in the workflow: generate first, then evaluate (the shared scoring utility used by all three operations), then aggregate, then refine.

In [4]:
def generate_thoughts(problem: str, parent_node: ThoughtNode, num_thoughts: int = 3) -> List[str]:
    """
    Generate multiple concrete candidate next reasoning steps from a given node.

    Args:
        problem: The original problem statement.
        parent_node: The node we are expanding from.
        num_thoughts: How many alternative next steps to request.

    Returns:
        A list of candidate thought strings.
    """
    # Reconstruct the full reasoning chain from root to this node for context
    path = parent_node.get_path()
    if len(path) <= 1:
        # Only the root problem node is on the path — we are at the very first step
        context = "No reasoning steps yet — this is the very first step."
    else:
        # Show the complete chain so the model sees the full sequence leading here
        context = "Reasoning so far:\n" + "\n".join(
            f"{i+1}. {step}" for i, step in enumerate(path)
        )

    system_msg = SystemMessage(content=(
        f"You are a systematic problem solver working step by step. "
        f"Generate {num_thoughts} CONCRETE, SPECIFIC next reasoning steps. "
        f"Each step must directly perform a deduction or calculation — "
        f"not describe what should be done, but actually do it. "
        f"Each of the {num_thoughts} steps should take a distinctly different angle. "
        f"Respond with ONLY a JSON array of strings, no other text."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"{context}\n\n"
        f"What are {num_thoughts} possible concrete next steps?"
    ))

    response = llm.invoke([system_msg, user_msg])

    # Strip any markdown code fence the model may have wrapped around the JSON
    raw = response.content.strip()
    if raw.startswith('```'):
        lines = raw.split('\n')
        raw = '\n'.join(lines[1:])               # drop the opening ```json line
        if raw.rstrip().endswith('```'):
            raw = raw[:raw.rstrip().rfind('```')]  # drop the closing ```
        raw = raw.strip()

    # First attempt: parse as a JSON array
    try:
        thoughts = json.loads(raw)
        if isinstance(thoughts, list):
            # Strip extra quotes the model occasionally wraps around individual items
            return [str(t).strip('"').strip("'") for t in thoughts[:num_thoughts]]
    except json.JSONDecodeError:
        pass  # fall through to line-by-line fallback

    # Fallback: split by line, skip bare JSON punctuation, strip numbering and bullets
    artifacts = {'[', ']', '{', '}', '```', '```json'}
    lines = [line.strip() for line in response.content.split('\n') if line.strip()]
    cleaned = []
    for line in lines:
        if line in artifacts or line.startswith('```'):
            continue
        if line and line[0].isdigit():          # strip "1." or "1)" list prefixes
            line = line.split('.', 1)[-1].split(')', 1)[-1].strip()
        line = line.lstrip('- ').lstrip('* ').strip('"').strip("'")
        if line:
            cleaned.append(line)
    return cleaned[:num_thoughts]

**`generate_thoughts`** reconstructs the full reasoning chain from root to `parent_node` using `get_path()`, then passes it to the model as numbered context - identical to the Tree of Thoughts approach. The only design difference is that the function takes a `ThoughtNode` directly rather than a pre-built path list, which is cleaner because the caller does not need to maintain the path separately.

The JSON parsing is the same three-tier approach used in Tree of Thoughts: strip any markdown code fence first, attempt `json.loads` on the cleaned text, fall back to line splitting if that fails, and filter bare JSON punctuation tokens (`[`, `]`, `` ``` ``) that would otherwise appear as spurious thoughts.

In [5]:
# Quick test: generate three opening thoughts for a sample problem
test_problem = "Design a notification system that alerts users without overwhelming them."

root_test = ThoughtNode(thought=f"Problem: {test_problem}", score=5.0, operation="root")
initial_thoughts = generate_thoughts(test_problem, root_test, num_thoughts=3)

print("Generated initial thoughts:")
for i, t in enumerate(initial_thoughts, 1):
    print(f"  {i}. {t}")

Generated initial thoughts:
  1. Identify the types of notifications needed by conducting a survey with potential users to gather their preferences and priorities.
  2. Analyze existing notification systems to determine their frequency and user engagement rates to establish a baseline for optimal notification intervals.
  3. Create a prototype of the notification settings interface allowing users to customize the types and frequencies of alerts they wish to receive.


## Evaluate thought quality
The evaluation function is identical in purpose to the one in Tree of Thoughts: give the model a thought in context and ask it to return a score from 0 to 10 using the same rubric of concrete progress, logical soundness, and coherent build on prior steps. The one addition is a third context format - when `context_nodes` contains multiple nodes (as it will when evaluating an aggregate thought), the function lists each independently rather than constructing a sequential chain. This matches the way aggregate context is framed in the generation and aggregation prompts, so the evaluator sees the same framing that the generator used.

In [6]:
def evaluate_thought(problem: str, thought: str, context_nodes: List[ThoughtNode]) -> float:
    """
    Score a thought on a 0–10 scale using the LLM as a judge.

    Args:
        problem: The original problem statement.
        thought: The thought to score.
        context_nodes: Parent nodes that provide context. For aggregate nodes this
                       will be several nodes from independent branches.

    Returns:
        A float in [0.0, 10.0].
    """
    # Build context to match the framing used during generation
    if not context_nodes:
        context = "This is the first reasoning step."
    elif len(context_nodes) == 1:
        # Single parent: show the full reasoning chain leading here
        path = context_nodes[0].get_path()
        context = "Prior reasoning:\n" + "\n".join(
            f"{i+1}. {step}" for i, step in enumerate(path)
        )
    else:
        # Multiple parents (aggregate): present them as parallel independent inputs
        context = "Synthesised from these parallel thoughts:\n" + "\n".join(
            f"  - {node.thought}" for node in context_nodes
        )

    system_msg = SystemMessage(content=(
        "You are an expert evaluator of reasoning steps. "
        "Score the following thought 0–10 based on: "
        "(1) does it concretely move toward solving the problem, "
        "(2) is the reasoning logically sound, "
        "(3) does it build coherently on the prior context? "
        "Respond with ONLY a single number between 0 and 10."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"{context}\n\n"
        f"Thought to evaluate: {thought}\n\nScore (0–10):"
    ))

    response = llm.invoke([system_msg, user_msg])

    try:
        raw = response.content.strip()
        raw = raw.split('/')[0]     # handle "8/10" format
        raw = raw.split(':')[-1]    # handle "Score: 8" format
        return max(0.0, min(10.0, float(raw.strip())))
    except (ValueError, AttributeError):
        return 5.0  # neutral fallback when parsing fails

**`evaluate_thought`** adds a multi-parent context branch to handle the case where an aggregate node is being scored. Presenting the parent thoughts as a bullet list - rather than a numbered sequential chain - correctly signals to the evaluator that these are independent parallel inputs. The score parsing, clamping to `[0, 10]`, and `5.0` neutral fallback are identical to the Tree of Thoughts version.

In [7]:
# Test: score the first generated thought against the root as context
sample_thought = initial_thoughts[0]
score = evaluate_thought(test_problem, sample_thought, [root_test])

print(f"Thought: {sample_thought}")
print(f"Score:   {score:.1f} / 10")

Thought: Identify the types of notifications needed by conducting a survey with potential users to gather their preferences and priorities.
Score:   8.0 / 10


## Aggregate: the operation that defines GoT
Aggregation is what makes Graph of Thoughts structurally different from Tree of Thoughts. In a tree, information only flows downward along separate branches - they never converge. In a graph, we can take the outputs of several independent branches and combine them into a single node that draws on all of them simultaneously.

The aggregate operation receives a list of nodes - typically the terminal nodes of parallel branches — passes all their thoughts to the model as labelled independent inputs, and asks for one synthesised response. The result becomes a new node whose `parents` list contains every input node. In the graph this appears as multiple incoming edges converging on a single node - a fan-in pattern that cannot exist in a tree.

The prompt design for aggregation differs from generation in one important way: inputs are presented as "Branch 1, Branch 2, ..." rather than as numbered sequential steps. This framing explicitly signals to the model that the inputs are parallel, not sequential — which reduces the frequency of outputs that simply continue the last branch rather than integrating all of them.

In [8]:
def aggregate_thoughts(problem: str, nodes_to_merge: List[ThoughtNode]) -> Tuple[str, float]:
    """
    Combine the thoughts from multiple nodes into one synthesised thought.
    This is the defining GoT operation: the result node will have multiple parents.

    Args:
        problem: The original problem statement.
        nodes_to_merge: The nodes whose thoughts will be combined.

    Returns:
        (synthesised_thought, score) for the new aggregate node.
    """
    # Label each input as a "Branch" — not a numbered step — to signal parallelism
    branch_summary = "\n".join(
        f"Branch {i+1}: {node.thought}" for i, node in enumerate(nodes_to_merge)
    )

    system_msg = SystemMessage(content=(
        "You are synthesising the conclusions of several independent, parallel reasoning branches. "
        "Produce one coherent, comprehensive answer that integrates all the inputs. "
        "Do not just list them — weave them into a unified response. "
        "Respond with ONLY the synthesised thought as plain text."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"Independent branch conclusions:\n{branch_summary}\n\n"
        f"Synthesised answer:"
    ))

    response = llm.invoke([system_msg, user_msg])
    thought = response.content.strip()

    # Evaluate the synthesised thought with all parent nodes as context
    score = evaluate_thought(problem, thought, nodes_to_merge)
    return thought, score

**`aggregate_thoughts`** labels each input as "Branch 1:", "Branch 2:", etc. rather than numbering them as sequential steps. The instruction to "weave them into a unified response" rather than list them is the key phrase that discourages naive concatenation. After generation, the result is scored with `evaluate_thought` using all parent nodes as context, which exercises the multi-parent branch of the evaluator.

In [9]:
# Quick test: aggregate two manually constructed nodes to see the synthesis
node_a = ThoughtNode(
    thought="Rate-limit notifications to a maximum of 3 per hour per channel.",
    parents=[root_test], score=7.5, operation="generate"
)
node_b = ThoughtNode(
    thought="Categorise notifications into urgency tiers: critical, standard, informational.",
    parents=[root_test], score=7.0, operation="generate"
)

agg_thought, agg_score = aggregate_thoughts(test_problem, [node_a, node_b])
print(f"Aggregated [{agg_score:.1f}/10]:")
print(agg_thought)

Aggregated [10.0/10]:
To create an effective notification system that alerts users without causing overwhelm, we can implement a structured approach that incorporates both rate limiting and urgency categorization. First, notifications will be limited to a maximum of three per hour per channel, ensuring that users are not bombarded with messages and can maintain focus on their tasks. Additionally, we will categorize notifications into three urgency tiers: critical notifications that require immediate attention, standard notifications that are important but not urgent, and informational notifications that provide useful updates without necessitating immediate action. This dual approach will not only help manage the frequency of alerts but also prioritize them based on importance, allowing users to engage with the most relevant notifications while reducing the likelihood of notification fatigue.


## Refine: iterative improvement
Refinement is the third GoT operation. It takes a single node and asks the model to produce one improved version of its thought - making it more specific, filling logical gaps, or correcting weaknesses. The result is a new node whose only parent is the node being refined, forming a simple chain of single-parent edges in the graph.

What makes refinement different from a regular `generate` step is the prompt framing: the model is shown the current thought *alongside its evaluation score* and asked explicitly to improve it, rather than being asked to explore new directions. This reliably produces incremental improvements rather than lateral moves.

Refinement is used in a short loop after aggregation. The `is_complete` function acts as an early-exit guard: before each refinement round it asks the model whether the current thought already constitutes a complete, specific final answer. If it does, the loop stops immediately rather than running to the iteration cap. The same "concrete specifics - not a plan" criterion used for `is_solution` in Tree of Thoughts is applied here.

In [10]:
def refine_thought(problem: str, node: ThoughtNode) -> Tuple[str, float]:
    """
    Produce one improved version of an existing node's thought.

    Args:
        problem: The original problem statement.
        node: The node to refine.

    Returns:
        (improved_thought, score).
    """
    system_msg = SystemMessage(content=(
        "You are improving an existing reasoning step. "
        "Make the thought more specific, fill any logical gaps, and ensure it directly answers the problem. "
        "Build on what is already there — do not start from scratch. "
        "Respond with ONLY the improved thought as plain text."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"Current thought (score {node.score:.1f}/10):\n{node.thought}\n\n"
        f"Improved thought:"
    ))

    response = llm.invoke([system_msg, user_msg])
    thought = response.content.strip()
    # Evaluate the improved thought against the original node as context
    score = evaluate_thought(problem, thought, [node])
    return thought, score


def is_complete(problem: str, node: ThoughtNode) -> bool:
    """
    Ask the LLM whether the node's thought is already a complete, specific final answer.
    Returns True only when the thought contains concrete specifics — not a plan or strategy.
    """
    system_msg = SystemMessage(content=(
        "You are checking whether a reasoning step is a final, complete answer. "
        "It must contain concrete specifics — actual values, steps, or recommendations — "
        "not a description of what to do next. "
        "Reply with ONLY 'yes' or 'no'."
    ))

    user_msg = HumanMessage(content=(
        f"Problem: {problem}\n\n"
        f"Thought:\n{node.thought}\n\n"
        f"Is this a complete, specific final answer?"
    ))

    response = llm.invoke([system_msg, user_msg])
    return response.content.strip().lower().startswith('yes')

**`refine_thought`** passes the current score in the prompt alongside the thought text. Showing the model that the thought was already evaluated and should be made *better* - not replaced - reliably produces incremental improvements.

**`is_complete`** applies the same "concrete specifics - actual values, steps, or recommendations" criterion used by `is_solution` in Tree of Thoughts. Checking only for concreteness (rather than correctness) keeps the check cheap and avoids the model second-guessing its own synthesis.

## Running the full Graph of Thoughts search
With the three operations in place, the driver function orchestrates them into the canonical GoT workflow. The pattern has four steps:
1. **Branch heads** - from the root, generate `num_branches` independent starting thoughts in a single `generate_thoughts` call. Each head is scored and becomes a separate branch.
2. **Branch extension** - extend each head by one further `generate` step, producing leaf nodes at depth 2. These carry more concrete content than the broad opening thoughts, giving the aggregation step richer material to synthesise.
3. **Aggregate** - call `aggregate_thoughts` on all leaf nodes. This creates the multi-parent edge: `agg_node.parents` will contain every leaf node, and `len(agg_node.parents) == num_branches`.
4. **Refine loop** - call `refine_thought` repeatedly until `is_complete` returns `True` or `max_refinements` is reached.

The log follows the same format as `run_tree_of_thoughts`: one status line per step, one line per new node showing ID and score, and an explicit parent list for the aggregate node so the fan-in is visible.

In [11]:
def run_graph_of_thoughts(
    problem: str,
    num_branches: int = 3,
    max_refinements: int = 3
) -> dict:
    """
    Execute the canonical GoT workflow: branch → extend → aggregate → refine.

    Args:
        problem: The problem to solve.
        num_branches: Number of parallel branches to generate and later merge.
        max_refinements: Maximum refinement iterations after aggregation.

    Returns:
        dict with the final node, all nodes in traversal order, and the root.
    """
    ThoughtNode.reset_counter()  # ensure Node #1 is always the root for this run

    print(f"Graph of Thoughts  |  branches={num_branches}  |  max_refinements={max_refinements}")
    print("=" * 65)

    # ── Step 1: root ──────────────────────────────────────────────────────────
    root = ThoughtNode(thought=f"Problem: {problem}", score=5.0, operation="root")
    print(f"\n[Root] Node #{root.node_id}  {root.thought[:70]}")

    # ── Step 2: generate num_branches independent branch heads from the root ──
    print(f"\n--- Step 2: generate {num_branches} independent branch heads ---")
    raw_thoughts = generate_thoughts(problem, root, num_thoughts=num_branches)
    branch_heads: List[ThoughtNode] = []
    for raw in raw_thoughts:
        score = evaluate_thought(problem, raw, [root])
        node = ThoughtNode(thought=raw, parents=[root], score=score, operation="generate")
        branch_heads.append(node)
        print(f"  Node #{node.node_id}  [score={score:.1f}]  {raw[:72]}")

    # ── Step 3: extend each branch head by one further generate step ──────────
    print(f"\n--- Step 3: extend each branch by one step ---")
    leaf_nodes: List[ThoughtNode] = []
    for head in branch_heads:
        # Each branch extends independently — parallel reasoning in concept
        next_thoughts = generate_thoughts(problem, head, num_thoughts=1)
        raw = next_thoughts[0] if next_thoughts else head.thought
        score = evaluate_thought(problem, raw, [head])
        leaf = ThoughtNode(thought=raw, parents=[head], score=score, operation="generate")
        leaf_nodes.append(leaf)
        print(f"  Node #{leaf.node_id} (from #{head.node_id})  [score={score:.1f}]  {raw[:65]}")

    # ── Step 4: aggregate all leaf nodes into one converging node ─────────────
    print(f"\n--- Step 4: aggregate {len(leaf_nodes)} leaf nodes ---")
    agg_thought, agg_score = aggregate_thoughts(problem, leaf_nodes)
    # parents=leaf_nodes creates the multi-parent edge that defines graph structure
    agg_node = ThoughtNode(
        thought=agg_thought, parents=leaf_nodes, score=agg_score, operation="aggregate"
    )
    parent_ids = [n.node_id for n in leaf_nodes]
    print(f"  Node #{agg_node.node_id}  [score={agg_score:.1f}]  parents={parent_ids}")
    print(f"  {agg_thought[:80]}")

    # ── Step 5: refine loop — improve until complete or limit reached ─────────
    print(f"\n--- Step 5: refine (max {max_refinements} rounds) ---")
    current = agg_node
    for i in range(1, max_refinements + 1):
        if is_complete(problem, current):
            print(f"  Round {i}: already complete — stopping early.")
            break
        refined_thought, refined_score = refine_thought(problem, current)
        current = ThoughtNode(
            thought=refined_thought, parents=[current], score=refined_score, operation="refine"
        )
        print(f"  Round {i}: Node #{current.node_id}  [score={refined_score:.1f}]  {refined_thought[:65]}")

    print("\n" + "=" * 65)
    print(f"Done. {ThoughtNode._counter} nodes total.  Final: Node #{current.node_id}  [score={current.score:.1f}]")

    # Collect all nodes by DFS from root (follow children links forward)
    all_nodes, visited, stack = [], set(), [root]
    while stack:
        n = stack.pop()
        if n.node_id not in visited:
            visited.add(n.node_id)
            all_nodes.append(n)
            stack.extend(n.children)

    return {"final": current, "all_nodes": all_nodes, "root": root}

Three structural points in `run_graph_of_thoughts` are worth noting.
- **Two generate passes** rather than one. The first creates `num_branches` heads at depth 1; the second extends each to depth 2. Leaf nodes at depth 2 carry more concrete content than the broad opening thoughts at depth 1, giving the aggregation step better material to synthesise.
- **`parents=leaf_nodes` on the aggregate node** is the line that creates the multi-parent edge. Every leaf node appears in `agg_node.parents`, so `len(agg_node.parents) == num_branches`. The `print_graph` visualization displays this fan-in as `(←3 parents)`.
- **The DFS traversal at the end** collects all nodes by following `children` links forward from the root, using a `visited` set to guard against revisiting shared nodes if the graph is extended in future with cycles or diamond sub-graphs.

## Solving a problem with Graph of Thoughts
The decompose–aggregate pattern works best when a problem has genuinely independent sub-aspects that can be reasoned about in parallel before being combined. A question about improving a software team process fits well: independent branches can each focus on a different dimension - process, tooling, culture — and the aggregation step weaves their conclusions into a coherent whole. After the run we print the full graph to confirm the multi-parent edge and the refinement spine.

In [12]:
problem = (
    "What are the most effective strategies for a software team "
    "to reduce the time it takes to review and merge pull requests?"
)

results = run_graph_of_thoughts(
    problem=problem,
    num_branches=3,
    max_refinements=3
)

# Display the final answer
final = results["final"]
print(f"\n{'='*65}")
print(f"Final answer  [Node #{final.node_id}  score={final.score:.1f}/10  depth={final.depth}]")
print(f"{'='*65}")
print(final.thought)

Graph of Thoughts  |  branches=3  |  max_refinements=3

[Root] Node #1  Problem: What are the most effective strategies for a software team to

--- Step 2: generate 3 independent branch heads ---
  Node #2  [score=8.0]  Analyze the average time taken for pull request reviews over the past mo
  Node #3  [score=7.0]  Calculate the percentage of pull requests that are merged within 24 hour
  Node #4  [score=8.0]  Evaluate the distribution of review times by comparing the fastest 10% o

--- Step 3: extend each branch by one step ---
  Node #5 (from #2)  [score=9.0]  Calculate the average time taken for pull request reviews by summ
  Node #6 (from #3)  [score=8.0]  Analyze the average time taken for each pull request to be review
  Node #7 (from #4)  [score=8.0]  Analyze the review times of the fastest 10% of pull requests and 

--- Step 4: aggregate 3 leaf nodes ---
  Node #8  [score=10.0]  parents=[5, 6, 7]
  To effectively reduce the time it takes to review and merge pull requests, a sof

The log output has four clearly labelled steps. Step 2 generates three independent branch heads from the root; step 3 shows each head extended with a `(from #N)` annotation tracing the parent link; step 4 prints the aggregate node with its full parent list so the fan-in is explicit; step 5 shows each refinement round with a monotonically increasing (or plateauing) score. The final node count and score are summarised at the end.

## Visualizing the graph structure
The graph structure only becomes fully clear when we see all nodes laid out in their hierarchy. The key thing to look for is the aggregate node: it is the single point where all branches converge, and it carries a `(←3 parents)` annotation that no other node in the graph has. The branches above it are the parallel generate chains; the refinement spine below it is a straight single-parent sequence. Seeing both together confirms that the graph is genuinely non-tree-shaped.

In [18]:
def print_graph(root: ThoughtNode):
    """
    Print the thought graph as an ASCII-art hierarchy.
    Aggregate nodes are annotated with (←N parents) to make the fan-in visible.
    A visited set prevents the aggregate node and its refinement subtree from being
    printed multiple times — once for each leaf that holds it as a child.
    """
    op_labels = {"root": "ROOT", "generate": "GEN", "aggregate": "AGG", "refine": "REF"}

    # Tracks node IDs that have already been fully rendered.
    # Without this, the aggregate node would be printed once per leaf it belongs to,
    # because the constructor registers it as a child of every parent.
    visited: set = set()

    def _render(node, prefix, is_last):
        branch = "└── " if is_last else "├── "
        label = op_labels.get(node.operation, "   ")
        # Annotate nodes with more than one parent to show the fan-in count
        fanin = f" (←{len(node.parents)} parents)" if len(node.parents) > 1 else ""

        if node.node_id in visited:
            # This node was already rendered under an earlier parent.
            # Print a back-reference instead of duplicating the entire subtree.
            print(f"{prefix}{branch}[#{node.node_id} {label} {node.score:.1f}]{fanin} ↑ already shown above")
            return   # do not recurse — children were already printed

        visited.add(node.node_id)   # mark before recursing to handle any future cycles

        # Truncate long thoughts so the line stays readable in a terminal
        preview = node.thought[:55] + "..." if len(node.thought) > 55 else node.thought
        print(f"{prefix}{branch}[#{node.node_id} {label} {node.score:.1f}]{fanin} {preview}")

        # Last child closes the vertical bar (spaces); earlier children keep it open (│)
        child_prefix = prefix + ("    " if is_last else "│   ")
        for i, child in enumerate(node.children):
            _render(child, child_prefix, is_last=(i == len(node.children) - 1))

    # Print the root without a branch character — it has no parent to branch from
    preview = root.thought[:70] + "..." if len(root.thought) > 70 else root.thought
    print(f"\n[#{root.node_id} ROOT] {preview}")
    visited.add(root.node_id)
    for i, child in enumerate(root.children):
        _render(child, prefix="", is_last=(i == len(root.children) - 1))


# Visualise the graph produced by the run above
print_graph(results["root"])


[#1 ROOT] Problem: What are the most effective strategies for a software team to...
├── [#2 GEN 8.0] Analyze the average time taken for pull request reviews...
│   └── [#5 GEN 9.0] Calculate the average time taken for pull request revie...
│       └── [#8 AGG 10.0] (←3 parents) To effectively reduce the time it takes to review and m...
│           └── [#9 REF 9.0] To effectively reduce the time it takes to review and m...
│               └── [#10 REF 10.0] To effectively reduce the time it takes to review and m...
│                   └── [#11 REF 10.0] To effectively reduce the time it takes to review and m...
├── [#3 GEN 7.0] Calculate the percentage of pull requests that are merg...
│   └── [#6 GEN 8.0] Analyze the average time taken for each pull request to...
│       └── [#8 AGG 10.0] (←3 parents) ↑ already shown above
└── [#4 GEN 8.0] Evaluate the distribution of review times by comparing ...
    └── [#7 GEN 8.0] Analyze the review times of the fastest 10% of pull req...
        

**`print_graph`** uses the same recursive prefix-building algorithm as `print_tree` in Tree of Thoughts. The one addition is the `fanin` annotation: any node with more than one parent gets `(←N parents)` appended to its label. In a standard run with three branches, only the aggregate node carries this annotation - making it immediately visible as the convergence point of the graph. The refinement chain below it shows as a straight single-parent spine extending from that point.

Graph of Thoughts extends Tree of Thoughts by adding one structural capability: a node can have multiple parents. This single change enables the decompose–aggregate pattern where parallel branches are reasoned about independently and then merged - something a tree structure cannot express.

**When Graph of Thoughts fits better than Tree of Thoughts:**
- The problem has independent sub-aspects that benefit from parallel exploration before synthesis.
- A single answer should integrate insights from multiple angles, not just pick the best single branch.
- Iterative sharpening after an initial synthesis is more useful than continued branching.

**Cost trade-offs:**
- For `num_branches=3, max_refinements=3` the workflow makes roughly 15–20 LLM calls: two generate passes (3 × 2 = 6 calls) plus scoring for each (6 more), one aggregate call, and up to 3 refine-and-score rounds.
- This is more than a shallow ToT run but less than a deep best-first search that keeps branching without converging.
- The aggregate call is the most expensive prompt (all branch texts as input) but produces only one output.